# 12: PowerPoint Generation Testing

This notebook tests PowerPoint generation features:

1. **Basic Presentation** - Title, content, metrics slides
2. **Analytics Presentation** - Charts, tables, insights
3. **DataFrame Presentation** - Auto-generated from pandas data
4. **Custom Presentation** - Full control over slide structure

For a real-world example using these capabilities to generate a branded enterprise onboarding deck (PPT + PDF) with charts, tables, and a convergence diagram, see **[Notebook 21: Enterprise Onboarding Presentation](./21_Enterprise_Onboarding_Presentation.ipynb)**.

## Prerequisites

```bash
pip install python-pptx
```

## What this shows
Assembling `Argument` objects into branded PPTX slides via the SAL-66 pipeline.

## Why it matters
Reports/decks now flow from `Chain` → `Argument` → Slides — shared primitive across PDF and PPTX.

## Prereqs
- `pip install 'siege-utilities[reporting,pptx]'`

## Next
- `18_Google_Workspace.ipynb` for Google Slides output; `28_Polling_Survey_Analysis.ipynb` for Argument sources.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from IPython.display import display

# Default guard — cell-3 overrides to True if python-pptx is available
PPTX_AVAILABLE = False

# Ensure siege_utilities is importable
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

print(f"Python: {sys.version}")
print("PowerPoint Generation notebook ready.")

In [ ]:
# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_12"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Check python-pptx availability
try:
    from pptx import Presentation
    from pptx.util import Inches, Pt
    print("python-pptx: available")
    PPTX_AVAILABLE = True
except ImportError as e:
    print(f"python-pptx: NOT available ({e})")
    print("Install with: pip install python-pptx")
    PPTX_AVAILABLE = False

In [ ]:
if not PPTX_AVAILABLE:
    print("WARNING: Skipping remaining cells — python-pptx not installed")
else:
    from siege_utilities.reporting.powerpoint_generator import PowerPointGenerator
    print("PowerPointGenerator imported successfully.")

## 1. Create Sample Data

In [ ]:
if PPTX_AVAILABLE:
    np.random.seed(42)
    
    performance_data = pd.DataFrame({
        'Quarter': ['Q1 2025', 'Q2 2025', 'Q3 2025', 'Q4 2025'],
        'Revenue ($M)': [12.5, 15.2, 18.7, 22.1],
        'Users (K)': [145, 189, 234, 278],
        'Conversion (%)': [3.2, 3.5, 4.1, 4.3]
    })
    
    regional_data = pd.DataFrame({
        'Region': ['Northeast', 'Southeast', 'Midwest', 'West', 'International'],
        'Revenue ($M)': [18.2, 12.5, 8.7, 22.4, 6.7],
        'Growth (%)': [12, 8, 15, 22, 35]
    })
    
    print("Performance Data:")
    display(performance_data)
    print("\nRegional Data:")
    display(regional_data)

## 2. Basic Analytics Presentation

In [ ]:
if PPTX_AVAILABLE:
    pptx_gen = PowerPointGenerator(
        client_name="TestClient",
        output_dir=OUTPUT_DIR
    )
    print(f"PowerPointGenerator initialized (client: {pptx_gen.client_name})")

In [ ]:
if PPTX_AVAILABLE:
    report_data = {
        'executive_summary': """
FY 2025 Performance Highlights:

- Total revenue of $68.5M (+15.4% YoY)
- 846 new customers acquired
- Conversion rate improved to 4.3%
- West region leads with 33% market share
        """,
        'metrics': {
            'Total Revenue': {'value': '$68.5M', 'status': '+15.4%'},
            'New Customers': {'value': '846', 'status': '+23%'},
            'Conversion Rate': {'value': '4.3%', 'status': '+0.8pp'},
            'Customer Satisfaction': {'value': '4.7/5', 'status': 'Excellent'},
            'Net Promoter Score': {'value': '72', 'status': '+5'},
            'Churn Rate': {'value': '2.1%', 'status': '-0.3pp'}
        },
        'charts': [
            {'title': 'Revenue by Quarter', 'type': 'bar'},
            {'title': 'Regional Distribution', 'type': 'pie'}
        ],
        'tables': [
            {
                'title': 'Quarterly Performance',
                'headers': performance_data.columns.tolist(),
                'data': performance_data.values.tolist()
            }
        ],
        'insights': [
            'West region shows highest growth potential',
            'Q3 conversion spike correlates with product launch',
            'International expansion driving new customer growth',
            'Customer satisfaction at all-time high'
        ]
    }
    
    print(f"Report data: {len(report_data['metrics'])} metrics, {len(report_data['insights'])} insights")

In [ ]:
if PPTX_AVAILABLE:
    try:
        analytics_pptx = pptx_gen.create_analytics_presentation(
            report_data=report_data,
            presentation_title="FY 2025 Analytics Report",
            include_charts=True,
            include_tables=True
        )
        
        if analytics_pptx.exists():
            size_kb = analytics_pptx.stat().st_size / 1024
            print(f"Analytics presentation generated: {analytics_pptx.name} ({size_kb:.1f} KB)")
        else:
            print("WARNING: Presentation was not created")
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()

## 3. Performance Presentation

In [ ]:
if PPTX_AVAILABLE:
    perf_data = {
        'overview': """
Performance Overview for FY 2025:

The company exceeded all key performance targets 
with strong growth across all regions.
        """,
        'revenue': {'current': '$68.5M', 'target': '$65.0M', 'achievement': '105%'},
        'customers': {'current': '8,460', 'target': '7,500', 'achievement': '113%'},
        'satisfaction': {'current': '4.7', 'target': '4.5', 'achievement': '104%'},
        'trends': {
            'Revenue': 'Upward trajectory',
            'Customer Growth': 'Accelerating'
        },
        'recommendations': [
            'Increase investment in West region',
            'Launch customer loyalty program',
            'Expand product line for enterprise segment',
            'Accelerate international expansion'
        ]
    }
    
    metrics_list = ['revenue', 'customers', 'satisfaction']
    print("Performance data prepared.")

In [ ]:
if PPTX_AVAILABLE:
    try:
        perf_pptx = pptx_gen.create_performance_presentation(
            performance_data=perf_data,
            metrics=metrics_list,
            presentation_title="FY 2025 Performance Review"
        )
        
        if perf_pptx.exists():
            size_kb = perf_pptx.stat().st_size / 1024
            print(f"Performance presentation generated: {perf_pptx.name} ({size_kb:.1f} KB)")
        else:
            print("WARNING: Presentation was not created")
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()

## 4. DataFrame Presentation

In [ ]:
if PPTX_AVAILABLE:
    sample_df = pd.DataFrame({
        'Product': [f'Product_{i}' for i in range(1, 21)],
        'Sales': np.random.randint(1000, 10000, 20),
        'Revenue': np.random.uniform(10000, 100000, 20).round(2),
        'Growth': np.random.uniform(-10, 30, 20).round(1),
        'Rating': np.random.uniform(3.5, 5.0, 20).round(1)
    })
    
    print(f"Sample DataFrame: {len(sample_df)} rows, {len(sample_df.columns)} columns")
    display(sample_df.head(10))

In [ ]:
if PPTX_AVAILABLE:
    try:
        df_pptx = pptx_gen.create_presentation_from_dataframe(
            df=sample_df,
            presentation_title="Product Performance Analysis",
            max_slides=8
        )
        
        if df_pptx.exists():
            size_kb = df_pptx.stat().st_size / 1024
            print(f"DataFrame presentation generated: {df_pptx.name} ({size_kb:.1f} KB)")
        else:
            print("WARNING: Presentation was not created")
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()

## 5. Custom Presentation with Full Control

In [ ]:
if PPTX_AVAILABLE:
    custom_config = {
        'type': 'quarterly_review',
        'title': 'Q4 2025 Business Review',
        'slides': [
            {
                'type': 'content', 'title': 'Agenda',
                'content': '1. Executive Summary\n2. Financial Performance\n3. Market Analysis\n4. Strategic Initiatives\n5. Q1 2026 Outlook'
            },
            {
                'type': 'content', 'title': 'Executive Summary',
                'content': 'Q4 2025 was our strongest quarter:\n\n- Revenue: $22.1M (+18% QoQ)\n- New customers: 278\n- Market share: 33% in West\n- Customer NPS: 72'
            },
            {
                'type': 'table', 'title': 'Regional Performance',
                'headers': regional_data.columns.tolist(),
                'data': regional_data.values.tolist()
            },
            {
                'type': 'comparison', 'title': 'YoY Comparison',
                'left_content': 'Q4 2024:\n- Revenue: $18.7M\n- Customers: 6,200\n- NPS: 67',
                'right_content': 'Q4 2025:\n- Revenue: $22.1M\n- Customers: 8,460\n- NPS: 72'
            },
            {
                'type': 'content', 'title': 'Next Steps',
                'content': 'Q1 2026 Priorities:\n\n1. Launch enterprise product tier\n2. Expand to European markets\n3. Increase marketing spend 20%\n4. Hire 15 new engineers'
            }
        ]
    }
    
    print(f"Custom presentation config: {len(custom_config['slides'])} slides")

In [ ]:
if PPTX_AVAILABLE:
    try:
        custom_pptx = pptx_gen.create_custom_presentation(custom_config)
        
        if custom_pptx.exists():
            size_kb = custom_pptx.stat().st_size / 1024
            print(f"Custom presentation generated: {custom_pptx.name} ({size_kb:.1f} KB)")
        else:
            print("WARNING: Presentation was not created")
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
if PPTX_AVAILABLE:
    print("=" * 50)
    print("POWERPOINT GENERATION TEST RESULTS")
    print("=" * 50)
    
    generated_files = [
        ('Analytics PPTX', analytics_pptx),
        ('Performance PPTX', perf_pptx),
        ('DataFrame PPTX', df_pptx),
        ('Custom PPTX', custom_pptx),
    ]
    
    results = []
    for name, path in generated_files:
        if path.exists():
            size_kb = path.stat().st_size / 1024
            print(f"  PASS  {name}: {size_kb:.1f} KB")
            results.append(True)
        else:
            print(f"  FAIL  {name}: Not generated")
            results.append(False)
    
    passed = sum(results)
    total = len(results)
    print(f"\n{passed}/{total} presentations generated successfully.")
    print(f"Output directory: {OUTPUT_DIR}")
    
    if passed == total:
        print("\nAll PowerPoint features working!")
    else:
        print("\nSome features need attention.")
else:
    print("PowerPoint tests skipped — python-pptx not installed.")

---

## Part 2: Google Workspace — Docs, Sheets, Slides

Same `Argument` inputs, different output surface. Google Slides is the canonical target; Docs and Sheets are demoed as peers where they accept the same structured data.


In [ ]:
"""Setup: imports and authentication."""

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Check Google API availability
try:
    import googleapiclient  # noqa: F401
    import google.auth  # noqa: F401
    GOOGLE_AVAILABLE = True
    print("Google API libraries available")
except ImportError:
    GOOGLE_AVAILABLE = False
    print("Google API libraries NOT installed — pip install siege-utilities[analytics]")

# Workspace client
from siege_utilities.analytics.google_workspace import GoogleWorkspaceClient

# Sheets API
from siege_utilities.analytics.google_sheets import (
    create_spreadsheet, write_values, append_rows, read_values,
    read_dataframe, write_dataframe, add_sheet,
    get_spreadsheet_metadata, copy_spreadsheet,
)

# Slides API
from siege_utilities.analytics.google_slides import (
    create_presentation, get_presentation, copy_presentation,
    add_blank_slide, create_textbox, insert_text as slides_insert_text,
)

# Docs API
from siege_utilities.analytics.google_docs import (
    create_document, get_document, copy_document,
    read_document_text, insert_text as docs_insert_text,
    insert_paragraph, insert_table, replace_text,
)

# Multi-account management
from siege_utilities.config import GoogleAccountRegistry
from siege_utilities.config.models.google_account import (
    GoogleAccount, GoogleAccountType, GoogleAccountStatus,
)
from siege_utilities.config.models.person import Person

print("All imports successful")

## 1. Setup & Authentication

In [ ]:
"""Authenticate via 1Password.

from_1password() auto-detects the credential type:
  - Service account JSON → server-to-server (no browser)
  - OAuth client secret  → installed-app flow (browser on first use)

1Password layout (Siege_Analytics = TLTQ3ANAABGCNEK7KIAOTDNK2Q):
  - Service account: vault=Employee, item="Google Service Account - siege_utilities"
  - OAuth client:    vault=Personal, item="Google OAuth Client - siege_utilities"
"""

# Siege_Analytics account UUID (portable across machines)
OP_ACCOUNT = "TLTQ3ANAABGCNEK7KIAOTDNK2Q"

client = None

# --- Default: Service account (server-to-server, no browser) ---
try:
    client = GoogleWorkspaceClient.from_1password(
        item_title="Google Service Account - siege_utilities",
        vault="Employee",
        account=OP_ACCOUNT,
    )
    print("Authenticated via service account — write operations enabled")
except Exception as e:
    print(f"Service account auth failed: {e}")

# --- Fallback: OAuth (browser required on first use) ---
if client is None:
    try:
        client = GoogleWorkspaceClient.from_1password(
            item_title="Google OAuth Client - siege_utilities",
            vault="Personal",
            account=OP_ACCOUNT,
        )
        print("Authenticated via OAuth — write operations enabled")
    except Exception as e:
        print(f"OAuth auth also failed: {e}")
        print("Running in read-only demo mode (API calls skipped)")

## Inline Demo Data — elect.info Onboarding Content

All data below is from the elect.info enterprise onboarding materials. Hardcoded here
for notebook portability (no external file dependencies).

In [ ]:
"""elect.info onboarding data — inline for portability."""

DATA_SOURCES = [
    ("FEC Electronic Filings", "Campaign Finance", 2000, 2026, "Active", "2.4 TB"),
    ("FEC Legacy/Bulk", "Campaign Finance", 1980, 1999, "Active", "100M+ records"),
    ("Individual Contributions", "Campaign Finance", 1980, 2026, "Active", "255M+ records"),
    ("TX Ethics Commission", "Campaign Finance", 2000, 2026, "Planned", "3M+ records"),
    ("FL Elections", "Campaign Finance", 1996, 2026, "Planned", "30M+ records"),
    ("Decennial Census", "Demographic", 2000, 2020, "Active", "330M records"),
    ("ACS 1/3/5Y", "Demographic", 2009, 2026, "Active", "1000+ variables"),
    ("PL 94-171", "Demographic", 2020, 2020, "Active", "Block-level"),
    ("TIGER/Line", "Geographic", 2010, 2026, "Active", "21 boundary types"),
    ("GADM", "Geographic", 2020, 2026, "Active", "6 admin levels"),
    ("NCES Districts", "Geographic", 2020, 2026, "In Dev", "13.5K districts"),
    ("NLRB Regions", "Geographic", 2020, 2026, "Active", "26 regions"),
]

DATA_SOURCES_HEADER = ["Source", "Category", "Start Year", "End Year", "Status", "Magnitude"]

SOURCE_INVENTORY = [
    ["Source", "Type", "Data Class", "Current/Planned", "Magnitude/Notes"],
    ["FEC Electronic Filings", "Tabular", "Campaign Finance", "Current", "2.4 TB electronic filings"],
    ["FEC Legacy/Bulk", "Tabular", "Campaign Finance", "Current", "100M+ historical records"],
    ["Individual Contributions", "Tabular", "Campaign Finance", "Current", "255M+ records processed"],
    ["Texas Ethics Commission", "Tabular", "State Campaign Finance", "Planned/In Progress", "3M+ records"],
    ["Florida Division of Elections", "Tabular", "State Campaign Finance", "Planned/In Progress", "30M+ records"],
    ["State Voter Files", "Tabular", "Voter Registration/Participation", "Planned", "10-30M records per state"],
    ["Decennial Census (Census Bureau)", "Tabular", "Demographic Context", "Current", "330M records"],
    ["ACS 1/3/5Y (Census Bureau)", "Tabular", "Demographic + Econometric Context", "Current", "1000+ variables"],
    ["PL 94-171", "Tabular", "Demographic Context", "Current", "Block-level population detail"],
    ["Redistricting Data Hub", "Tabular + Spatial", "District/precinct context", "Current/In Progress", "CVAP + boundary datasets"],
    ["TIGER/Line (Census Bureau)", "Spatial", "Boundaries/Geography", "Current", "21 boundary types"],
    ["GADM", "Spatial", "Administrative Geography", "Current", "6 admin levels"],
    ["NCES Districts", "Spatial", "Education Geography", "In Development", "13.5K districts"],
    ["NLRB Regions", "Spatial", "Labor Geography", "Current", "26 regions"],
]

DAILY_JOB_CHAIN = [
    ["Daily Job", "Purpose", "Result"],
    ["FecDownload", "Pull latest filings (daily mode)", "Fresh raw filings landed"],
    ["FecParse", "Parse filings into structured records", "Machine-readable records + manifest"],
    ["FecLoad", "Load into Silver and register for query", "Curated operational tables"],
    ["EnterprisePipeline", "Promote to Gold/Platinum and bridge", "Downstream-ready datasets"],
]

DATA_PRODUCT_CHAIN = [
    ["Layer/Product", "What It Is", "Who Relies On It"],
    ["Bronze", "Raw ingest history and replay source", "Ops + QA"],
    ["Silver", "Typed/validated records", "QA + Data Engineering"],
    ["Gold", "Entity-centered tables", "Product + Analytics"],
    ["Platinum", "Enriched/consumption layer", "Marketing + Product"],
    ["Bridge tables", "elect.info-compatible output schema", "Existing elect.info consumers"],
]

TWO_SYSTEM_TABLE = [
    ["Track", "Primary Purpose", "Cadence", "Output Role", "Flow Responsibility"],
    [
        "Prototype/Reconciliation",
        "Historical + bulk normalization and cross-system entity reconciliation",
        "Backfill and batch windows",
        "Foundational quality: reconciled entities and stable reference history",
        "Owns deep backfill, reconciliation QA, and schema-hardening before live promotion",
    ],
    [
        "Live Operations",
        "Daily operationalized processing for current filings and updates",
        "Daily scheduled runs",
        "Current production datasets, telemetry, and downstream delivery",
        "Owns daily flow: download -> parse -> load -> promote -> bridge delivery",
    ],
]

SOURCE_PURPOSE = [
    ["Source", "Why It Exists", "What It Contributes to Final Product"],
    ["FEC electronic + legacy", "Core filings + history", "Contribution events, committees, candidate finance behavior"],
    ["Census + ACS", "Population and socioeconomic context", "Demographic overlays for segmentation and shift analysis"],
    ["Geographic boundaries", "Place-level attachment", "District/county/tract level spatial intelligence"],
    ["State campaign finance", "Beyond federal scope", "State-level expansion of values and funding signals"],
]

TECH_TABLE = [
    ["Technology", "What It Is", "Why It Is In This Program"],
    ["Python", "Primary application and pipeline language", "Fast iteration, broad data ecosystem"],
    ["Spark", "Distributed compute engine", "Processes high-volume historical and daily files"],
    ["Warehouse layers (Hive/Delta + PostgreSQL)", "Curated Bronze/Silver/Gold/Platinum storage", "Separates raw ingest from trusted outputs"],
    ["PostgreSQL", "Relational system for serving curated datasets", "Stable transactional querying and API access"],
    ["Rundeck", "Job scheduler and orchestration", "Runs daily pipelines with auditable schedules"],
    ["Hydra + Pydantic", "Configuration and schema validation", "Prevents parser drift, enforces contracts"],
    ["Django API", "Service layer for structured outputs", "Delivers reusable datasets to products and consumers"],
    ["siege_utilities", "Branding, charting, and document generation", "Consistent PDF/PPT reporting artifacts"],
]

# Track file IDs for cleanup at the end
CREATED_FILE_IDS = []

print(f"Loaded {len(DATA_SOURCES)} data sources, {len(SOURCE_INVENTORY)-1} inventory rows")

## 2. Google Sheets — elect.info Data Source Inventory

Create a spreadsheet with the full data source inventory, demonstrate tab management,
DataFrame round-tripping, and append operations.

In [ ]:
"""2a. Create spreadsheet and write DATA_SOURCES to the first tab."""

if client:
    # Create spreadsheet with a named first tab
    spreadsheet_id = create_spreadsheet(client, "elect.info Data Sources", sheet_names=["Sources"])
    CREATED_FILE_IDS.append(spreadsheet_id)
    print(f"Created: {client.spreadsheet_url(spreadsheet_id)}")

    # Write header + data rows
    rows = [DATA_SOURCES_HEADER] + [list(row) for row in DATA_SOURCES]
    write_values(client, spreadsheet_id, "Sources!A1", rows)
    print(f"Wrote {len(rows)} rows (incl. header) to 'Sources' tab")
else:
    spreadsheet_id = None
    print("SKIPPED (no credentials) — would create 'elect.info Data Sources' spreadsheet")

In [ ]:
"""2b. Add a second tab with the detailed SOURCE_INVENTORY."""

if client and spreadsheet_id:
    # Add a new tab
    add_sheet(client, spreadsheet_id, "Full Inventory")
    write_values(client, spreadsheet_id, "'Full Inventory'!A1", SOURCE_INVENTORY)
    print(f"Added 'Full Inventory' tab with {len(SOURCE_INVENTORY)} rows")
else:
    print("SKIPPED — would add 'Full Inventory' tab with 15 rows")

In [ ]:
"""2c. Write a pandas DataFrame and round-trip it back."""

import pandas as pd

df_sources = pd.DataFrame(
    [list(row) for row in DATA_SOURCES],
    columns=DATA_SOURCES_HEADER,
)
print("Local DataFrame:")
display(df_sources)

if client and spreadsheet_id:
    # Write DataFrame to a new tab
    add_sheet(client, spreadsheet_id, "DataFrame Tab")
    write_dataframe(client, spreadsheet_id, df_sources, sheet_name="DataFrame Tab")
    print("Wrote DataFrame to 'DataFrame Tab'")

    # Read it back
    df_round_trip = read_dataframe(client, spreadsheet_id, "'DataFrame Tab'")
    print(f"\nRound-trip DataFrame ({len(df_round_trip)} rows):")
    display(df_round_trip.head())
else:
    print("\nSKIPPED write/read — DataFrame shown locally above")

In [ ]:
"""2d. Append rows, get metadata, copy spreadsheet."""

if client and spreadsheet_id:
    # Append a new data source
    new_row = [["OpenStreetMap", "Geographic", 2007, 2026, "Planned", "Global coverage"]]
    append_rows(client, spreadsheet_id, "Sources", new_row)
    print("Appended 1 row to 'Sources' tab")

    # Get metadata
    meta = get_spreadsheet_metadata(client, spreadsheet_id)
    sheets = [s["properties"]["title"] for s in meta["sheets"]]
    print(f"Spreadsheet title: {meta['properties']['title']}")
    print(f"Tabs: {sheets}")

    # Copy the entire spreadsheet
    copy_id = copy_spreadsheet(client, spreadsheet_id, "elect.info Data Sources (Copy)")
    CREATED_FILE_IDS.append(copy_id)
    print(f"Copied spreadsheet → {copy_id}")
else:
    print("SKIPPED — would append row, show metadata, and copy spreadsheet")

## 3. Google Slides — elect.info Overview Deck

Create a presentation with title, pipeline, and technology stack slides.

In [ ]:
"""3a. Create presentation and title slide."""

if client:
    pres_id = create_presentation(client, "elect.info Enterprise Overview")
    CREATED_FILE_IDS.append(pres_id)
    print(f"Created: {client.presentation_url(pres_id)}")

    # The default presentation has one blank slide — use it as the title slide
    pres = get_presentation(client, pres_id)
    title_slide_id = pres["slides"][0]["objectId"]

    # Add title and subtitle textboxes
    create_textbox(
        client, pres_id, title_slide_id,
        "elect.info Enterprise Overview",
        left=50, top=80, width=600, height=60,
    )
    create_textbox(
        client, pres_id, title_slide_id,
        "Political finance transparency through cross-system tracking\n"
        "and place-based social context.",
        left=50, top=160, width=600, height=60,
    )
    print("Title slide populated")
else:
    pres_id = None
    print("SKIPPED — would create 'elect.info Enterprise Overview' presentation")

In [ ]:
"""3b. Data pipeline slide — daily job chain."""

if client and pres_id:
    slide_id = add_blank_slide(client, pres_id)

    create_textbox(
        client, pres_id, slide_id,
        "Daily Pipeline Chain",
        left=50, top=30, width=600, height=40,
    )

    # Format the job chain as readable text
    pipeline_text = "\n".join(
        f"{row[0]:25s} {row[1]:45s} → {row[2]}"
        for row in DAILY_JOB_CHAIN[1:]  # skip header
    )
    create_textbox(
        client, pres_id, slide_id,
        pipeline_text,
        left=50, top=90, width=620, height=200,
    )
    print("Pipeline slide added")
else:
    print("SKIPPED — would add pipeline slide with daily job chain")

In [ ]:
"""3c. Technology stack slide + copy presentation."""

if client and pres_id:
    slide_id = add_blank_slide(client, pres_id)

    create_textbox(
        client, pres_id, slide_id,
        "Technology Stack",
        left=50, top=30, width=600, height=40,
    )

    # Format data product chain as text block
    product_text = "\n".join(
        f"{row[0]:20s} │ {row[1]:45s} │ {row[2]}"
        for row in DATA_PRODUCT_CHAIN[1:]
    )
    create_textbox(
        client, pres_id, slide_id,
        product_text,
        left=30, top=90, width=660, height=250,
    )
    print("Technology stack slide added")

    # Copy the entire presentation
    pres_copy_id = copy_presentation(client, pres_id, "elect.info Overview (Copy)")
    CREATED_FILE_IDS.append(pres_copy_id)
    print(f"Copied presentation → {pres_copy_id}")
else:
    print("SKIPPED — would add tech stack slide and copy presentation")

## 4. Google Docs — elect.info Onboarding Guide

Create a document with headings, paragraphs, tables, and find/replace.

In [ ]:
"""4a. Create document and insert heading + intro paragraph.

Note: Docs API uses character-index insertion. We build the document
from bottom to top (inserting at index 1 each time) so content appears
in the correct order, OR we track the running index.
Here we use a running-index approach for clarity.
"""

if client:
    doc_id = create_document(client, "elect.info Onboarding Guide")
    CREATED_FILE_IDS.append(doc_id)
    print(f"Created: {client.document_url(doc_id)}")

    # Insert title heading at the start of the document body
    insert_paragraph(client, doc_id, "elect.info Onboarding Guide", index=1, heading="TITLE")

    # Read current length to find the next insertion point
    doc = get_document(client, doc_id)
    end_index = doc["body"]["content"][-1]["endIndex"] - 1

    insert_paragraph(
        client, doc_id,
        "This guide covers the elect.info data sources, pipeline architecture, "
        "and the two-track system design.",
        index=end_index,
    )
    print("Title + intro paragraph inserted")
else:
    doc_id = None
    print("SKIPPED — would create 'elect.info Onboarding Guide' document")

In [ ]:
"""4b. Insert data sources section with a table."""

if client and doc_id:
    # Get current end index
    doc = get_document(client, doc_id)
    idx = doc["body"]["content"][-1]["endIndex"] - 1

    insert_paragraph(client, doc_id, "Data Sources", index=idx, heading="HEADING_1")

    doc = get_document(client, doc_id)
    idx = doc["body"]["content"][-1]["endIndex"] - 1

    insert_paragraph(
        client, doc_id,
        "elect.info integrates data from federal, state, demographic, and geographic sources.",
        index=idx,
    )

    # Insert the SOURCE_PURPOSE table (header + 4 data rows = 5 rows, 3 cols)
    doc = get_document(client, doc_id)
    idx = doc["body"]["content"][-1]["endIndex"] - 1
    insert_table(client, doc_id, rows=len(SOURCE_PURPOSE), cols=3, index=idx)
    print(f"Inserted SOURCE_PURPOSE table ({len(SOURCE_PURPOSE)} rows x 3 cols)")
else:
    print("SKIPPED — would insert 'Data Sources' heading + SOURCE_PURPOSE table")

In [ ]:
"""4c. Insert two-track system section and demonstrate replace_text."""

if client and doc_id:
    doc = get_document(client, doc_id)
    idx = doc["body"]["content"][-1]["endIndex"] - 1

    insert_paragraph(client, doc_id, "Two-Track System Design", index=idx, heading="HEADING_1")

    doc = get_document(client, doc_id)
    idx = doc["body"]["content"][-1]["endIndex"] - 1

    # Insert the TWO_SYSTEM_TABLE (header + 2 data rows = 3 rows, 5 cols)
    insert_table(client, doc_id, rows=len(TWO_SYSTEM_TABLE), cols=5, index=idx)
    print(f"Inserted TWO_SYSTEM_TABLE ({len(TWO_SYSTEM_TABLE)} rows x 5 cols)")

    # Demonstrate find/replace: update a placeholder
    doc = get_document(client, doc_id)
    idx = doc["body"]["content"][-1]["endIndex"] - 1
    insert_paragraph(
        client, doc_id,
        "Last updated: {{UPDATE_DATE}}",
        index=idx,
    )
    replace_text(client, doc_id, "{{UPDATE_DATE}}", "2026-03-09")
    print("Replaced {{UPDATE_DATE}} → 2026-03-09")
else:
    print("SKIPPED — would insert two-track section + replace_text demo")

In [ ]:
"""4d. Verify document content and copy."""

if client and doc_id:
    text = read_document_text(client, doc_id)
    print(f"Document text length: {len(text)} chars")
    print(f"First 300 chars:\n{text[:300]}...")

    # Copy the document
    doc_copy_id = copy_document(client, doc_id, "elect.info Onboarding Guide (Copy)")
    CREATED_FILE_IDS.append(doc_copy_id)
    print(f"\nCopied document → {doc_copy_id}")
else:
    print("SKIPPED — would read document text and copy")

## 5. Drive Operations — Share and Organize

Using the files created above, demonstrate sharing and folder management.

In [ ]:
"""5. Drive operations: share_file, move_to_folder."""

if client and spreadsheet_id:
    # Share the spreadsheet with a collaborator (update email as needed)
    # client.share_file(spreadsheet_id, "collaborator@example.com", role="reader")
    # print("Shared spreadsheet with collaborator")

    # To move files into a folder, you need a folder ID:
    # client.move_to_folder(spreadsheet_id, "FOLDER_ID_HERE")

    print("Drive operations available:")
    print("  client.share_file(file_id, email, role='writer')")
    print("  client.move_to_folder(file_id, folder_id)")
    print("  client.copy_file(file_id, title)")
    print(f"\nFiles created this session: {len(CREATED_FILE_IDS)}")
    for fid in CREATED_FILE_IDS:
        print(f"  {fid}")
else:
    print("SKIPPED — no files to operate on")

## 6. Multi-Account Management

Demonstrates the `GoogleAccount` model, `GoogleAccountRegistry`, `Person` integration,
and factory methods for building clients from registered accounts.

In [ ]:
"""6a. Create GoogleAccount instances."""

# OAuth account (e.g., personal workspace)
oauth_account = GoogleAccount(
    google_account_id="personal-workspace",
    email="dheeraj@elect.info",
    display_name="Personal Workspace",
    account_type=GoogleAccountType.OAUTH,
    status=GoogleAccountStatus.ACTIVE,
    is_default=True,
    scopes_granted=[
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/documents",
        "https://www.googleapis.com/auth/presentations",
        "https://www.googleapis.com/auth/drive.file",
    ],
    oauth_integration_name="google-workspace",
    token_file="~/.siege/tokens/workspace_token.json",
)

# Service account (e.g., automation pipeline)
svc_account = GoogleAccount(
    google_account_id="pipeline-svc",
    email="pipeline@elect-info.iam.gserviceaccount.com",
    display_name="Pipeline Service Account",
    account_type=GoogleAccountType.SERVICE_ACCOUNT,
    status=GoogleAccountStatus.ACTIVE,
    service_account_ref="op://Infrastructure/google-sa/credential",
    notes="Used by Rundeck jobs for automated report generation",
)

print("OAuth account:", oauth_account.get_info())
print()
print("Service account:", svc_account.get_info())

In [ ]:
"""6b. GoogleAccountRegistry — register, list, default selection, persistence."""

import tempfile
from pathlib import Path

registry = GoogleAccountRegistry()

# Register accounts
registry.register(oauth_account)
registry.register(svc_account)

print(f"Registered accounts: {len(registry.list_accounts())}")
for acct in registry.list_accounts():
    default_marker = " (DEFAULT)" if acct.is_default else ""
    print(f"  {acct.google_account_id}: {acct.email} [{acct.account_type.value}]{default_marker}")

# Get default account
default = registry.get_default()
print(f"\nDefault account: {default.google_account_id}")

# Filter by type
svc_accounts = registry.list_by_type(GoogleAccountType.SERVICE_ACCOUNT)
print(f"Service accounts: {[a.google_account_id for a in svc_accounts]}")

# JSON persistence
tmp_path = Path(tempfile.mkdtemp()) / "google_accounts.json"
registry.save(tmp_path)
print(f"\nSaved registry to {tmp_path}")

# Load from file
registry2 = GoogleAccountRegistry(config_path=tmp_path)
print(f"Loaded registry: {len(registry2.list_accounts())} accounts")

In [ ]:
"""6c. Person integration — attach Google accounts to a Person."""

person = Person(
    person_id="dheeraj-chand",
    name="Dheeraj Chand",
    email="dheeraj@siege-analytics.com",
)

# Add Google accounts to the person
person.add_google_account(oauth_account)
person.add_google_account(svc_account)

print(f"Person: {person.name}")
print(f"Google accounts: {len(person.google_accounts)}")
for ga in person.google_accounts:
    print(f"  {ga.google_account_id}: {ga.email}")

# Get the default Google account
default_ga = person.get_default_google_account()
print(f"\nDefault Google account: {default_ga.google_account_id if default_ga else 'None'}")

# Factory method: build a client from a Person's default account
# (would authenticate if credentials were available)
# client = GoogleWorkspaceClient.from_account(default_ga, person=person)

# Factory method: build from the registry
# client = GoogleWorkspaceClient.from_registry(registry, person=person)

print("\nFactory patterns:")
print("  GoogleWorkspaceClient.from_account(account, person=person)")
print("  GoogleWorkspaceClient.from_registry(registry, person=person)")

In [ ]:
"""6d. Migration utility — migrate from legacy OAuthIntegration to GoogleAccount."""

from siege_utilities.config.google_account_registry import migrate_single_account
from siege_utilities.config.models.oauth_integration import OAuthIntegration, OAuthScope

# Create a legacy OAuthIntegration (the kind you'd already have for GA)
legacy_oauth = OAuthIntegration(
    name="google-analytics-legacy",
    provider="google",
    service="google_analytics",
    client_id="legacy-client-id.apps.googleusercontent.com",
    client_secret="legacy-secret-value-here",
    redirect_uri="http://localhost:8080",
    scopes=[OAuthScope.ANALYTICS, OAuthScope.READ],
)

# Migrate to a GoogleAccount
migrated = migrate_single_account(
    oauth_integration=legacy_oauth,
    google_account_id="migrated_ga_user",
    email="dheeraj@elect.info",
    display_name="Migrated GA Account",
)
print("Migrated OAuthIntegration -> GoogleAccount:")
print(f"  ID: {migrated.google_account_id}")
print(f"  Email: {migrated.email}")
print(f"  Display name: {migrated.display_name}")
print(f"  Type: {migrated.account_type.value}")
print(f"  Scopes: {migrated.scopes_granted}")
print(f"  OAuth integration ref: {migrated.oauth_integration_name}")

## 7. API Summary

| Module | Function | Purpose |
|--------|----------|---------|
| **google_workspace** | `GoogleWorkspaceClient.from_oauth()` | OAuth2 authentication |
| | `GoogleWorkspaceClient.from_service_account()` | Service account auth |
| | `GoogleWorkspaceClient.from_account()` | Auth from GoogleAccount model |
| | `GoogleWorkspaceClient.from_registry()` | Auth from registry default |
| | `.sheets_service()` / `.docs_service()` / `.slides_service()` | Get API service objects |
| | `.copy_file()` / `.share_file()` / `.move_to_folder()` | Drive utilities |
| **google_sheets** | `create_spreadsheet()` | Create new spreadsheet |
| | `write_values()` / `read_values()` | Raw cell read/write |
| | `write_dataframe()` / `read_dataframe()` | Pandas round-trip |
| | `append_rows()` | Append after existing data |
| | `add_sheet()` | Add a tab |
| | `get_spreadsheet_metadata()` / `copy_spreadsheet()` | Inspect/copy |
| **google_slides** | `create_presentation()` | Create new presentation |
| | `add_blank_slide()` | Add slide with layout |
| | `create_textbox()` | Create and populate text box |
| | `insert_text()` / `insert_image()` | Content insertion |
| | `get_presentation()` / `copy_presentation()` | Inspect/copy |
| **google_docs** | `create_document()` | Create new document |
| | `insert_paragraph()` | Insert text with heading style |
| | `insert_text()` | Insert with bold/italic/size |
| | `insert_table()` / `insert_image()` | Structured content |
| | `replace_text()` | Find/replace throughout doc |
| | `read_document_text()` / `copy_document()` | Read/copy |
| **google_account_registry** | `GoogleAccountRegistry` | Multi-account management |
| | `migrate_single_account()` | Legacy OAuthIntegration migration |
| **config.models** | `GoogleAccount` | Account data model |
| | `Person.add_google_account()` | Person integration |

## 8. Cleanup

Optional: delete all files created during this notebook session.

In [ ]:
"""Cleanup: delete all files created in this session.

Uncomment and run to clean up Google Drive.
"""

# if client and CREATED_FILE_IDS:
#     for file_id in CREATED_FILE_IDS:
#         try:
#             client.drive_service().files().delete(fileId=file_id).execute()
#             print(f"Deleted {file_id}")
#         except Exception as e:
#             print(f"Could not delete {file_id}: {e}")
#     CREATED_FILE_IDS.clear()
#     print("Cleanup complete")
# else:
#     print("Nothing to clean up")

print(f"Files created this session: {len(CREATED_FILE_IDS)}")
for fid in CREATED_FILE_IDS:
    print(f"  {fid}")
print("\nUncomment the block above to delete them from Google Drive.")

## Related
- Source: `siege_utilities/reporting/powerpoint_generator.py`, `siege_utilities/reporting/pages/`
- Tests: `tests/test_survey_render.py`
